# Fase 3: Diseño Base de Datos

In [1]:
import mysql.connector                 # Importamos las librarias necesarias
from mysql.connector import errorcode
import pandas as pd
import csv

In [2]:
df_hr = pd.read_csv("clean-data/clean_df_hr.csv")
df_hr.head()

,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_number,environment_satisfaction,gender,...,percent_salary_hike,performance_rating,relationship_satisfaction,total_working_years,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager
0,41,True,Travel_Rarely,Sales,1,2,Life Sciences,1,2,Female,...,11,3,1,8,0,1,6,4,0,5
1,49,False,Travel_Frequently,Research & Development,8,1,Life Sciences,2,3,Male,...,23,4,4,10,3,3,10,7,1,7
2,37,True,Travel_Rarely,Research & Development,2,2,Other,4,4,Male,...,15,3,2,7,3,3,0,0,0,0
3,33,False,Travel_Frequently,Research & Development,3,4,Life Sciences,5,4,Female,...,11,3,3,8,3,3,8,7,3,0
4,27,False,Travel_Rarely,Research & Development,2,1,Medical,7,1,Male,...,12,3,4,6,3,3,2,2,2,2


Para ver qué tipo de relaciones podemos hacer entre tablas, primero usamos group by: 

In [3]:
# Teníamos dudas de si la futura tabla llamada 'performance satisfaction' tiene una relación de uno a uno o de uno a muchos con la tabla de 'employees'
# Claramente la relación tiene que ser de uno a muchos. Hay varias personas que tienen la misma combinación de datos en todas las columnas.
df_hr.groupby(
    ["performance_rating", "job_involvement", "environment_satisfaction", "relationship_satisfaction", "job_satisfaction", "work_life_balance"]
).size().reset_index(name="num_personas").sort_values(by="num_personas", ascending=False)

,performance_rating,job_involvement,environment_satisfaction,relationship_satisfaction,job_satisfaction,work_life_balance,num_personas
323,3,3,3,3,3,3,15
380,3,3,4,3,4,2,13
366,3,3,4,2,3,3,13
337,3,3,3,4,3,3,12
327,3,3,3,3,4,3,12
...,...,...,...,...,...,...,...
634,4,4,1,2,2,3,1
635,4,4,1,2,4,3,1
8,3,1,1,4,3,3,1
7,3,1,1,4,2,3,1


In [4]:
# Teníamos dudas de si la futura tabla llamada 'compensation' tiene una relación de uno a uno o de uno a muchos con la tabla de 'employees'
# Actualmente, podríamos hacer una relación de 1 a 1, pero preferimos dejarla como uno a muchos, por si acaso en el futuro hay más de una persona que tenga la misma combinación de valores en las columnas.
df_hr.groupby(
    ["monthly_income", "monthly_rate", "percent_salary_hike"]
).size().reset_index(name="num_personas").sort_values(by="num_personas", ascending=False)

,monthly_income,monthly_rate,percent_salary_hike,num_personas
1469,19999.0,5678,14,1
0,1009.0,26999,11,1
1,1051.0,13493,15,1
2,1052.0,23384,22,1
3,1081.0,16019,13,1
...,...,...,...,...
18,1483.0,16102,14,1
17,1420.0,25233,13,1
16,1416.0,17258,13,1
15,1393.0,24852,12,1


Conexión a MySQL 

In [5]:
try:
    # nos conectamos al servidor de mysql, no especificamos schema porque vamos a crearlo
    
    cnx = mysql.connector.connect(
        user='root',
        password= input('Escribe aquí tu contraseña:'),
        host='127.0.0.1'
        # auth_plugin = 'mysql_native_password'
    )
    print("¡Conexión exitosa a MySQL!") #
except mysql.connector.Error as err:
    # Si es un error de acceso denegado (ej. contraseña o usuario incorrecto)
    if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
        print("Algo está mal con tu nombre de usuario o contraseña.") #
    # Si la base de datos no existe
    elif err.errno == errorcode.ER_BAD_DB_ERROR:
        print("La base de datos no existe.") #
    # Para cualquier otro tipo de error
    else:
        print(err) #
        print("Código de Error:", err.errno) #
        print("SQLSTATE", err.sqlstate) #
        print("Mensaje", err.msg) #

¡Conexión exitosa a MySQL!


Crear SChema y usarlo

In [6]:
mycursor = cnx.cursor() # Iniciamos el cursor
mycursor.execute("CREATE SCHEMA IF NOT EXISTS hr_ABC;") # creamos schema
mycursor.execute("USE hr_ABC")

In [7]:
# EJECUTAR SI NOS EQUIVOCAMOS PARA BORRAR TABLAS Y EMPEZAR DE CERO
mycursor.execute("SET FOREIGN_KEY_CHECKS = 0")

tablas = [
    "job_details",
    "career_history",
    "compensation",
    "performance_satisfaction"
]

for t in tablas:
    mycursor.execute(f"DROP TABLE IF EXISTS {t}")

mycursor.execute("SET FOREIGN_KEY_CHECKS = 1")
cnx.commit()

print("Tablas eliminadas.")

Tablas eliminadas.


Preparar DATAFRAMES AUXILIARES

In [8]:
df_job_details = df_hr[["department", "job_role", "job_level", "business_travel", "over_time"]].drop_duplicates().reset_index(drop=True)

df_job_details.head(3)

,department,job_role,job_level,business_travel,over_time
0,Sales,Sales Executive,2,Travel_Rarely,True
1,Research & Development,Research Scientist,2,Travel_Frequently,False
2,Research & Development,Laboratory Technician,1,Travel_Rarely,True


In [9]:
df_career_history =df_hr[["total_working_years","years_at_company","years_in_current_role","years_since_last_promotion","years_with_curr_manager","num_companies_worked", "training_times_last_year",
            "attrition"]].drop_duplicates().reset_index(drop=True)


df_career_history.head(3)

,total_working_years,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager,num_companies_worked,training_times_last_year,attrition
0,8,6,4,0,5,8,0,True
1,10,10,7,1,7,1,3,False
2,7,0,0,0,0,6,3,True


In [10]:
df_compensation = df_hr[["monthly_income", "monthly_rate", "percent_salary_hike"]].drop_duplicates().reset_index(drop=True)

df_compensation.head(3)

,monthly_income,monthly_rate,percent_salary_hike
0,5993.0,19479,11
1,5130.0,24907,23
2,2090.0,2396,15


In [11]:
df_performance_satisfaction = df_hr[["performance_rating","job_involvement","environment_satisfaction","relationship_satisfaction","job_satisfaction", "work_life_balance",
    ] ].drop_duplicates().reset_index(drop=True)

df_performance_satisfaction.head(3)

,performance_rating,job_involvement,environment_satisfaction,relationship_satisfaction,job_satisfaction,work_life_balance
0,3,3,2,1,4,1
1,4,2,3,4,2,3
2,3,2,4,2,3,3


In [12]:
# Convertir booleanas a números para insertar en MySQL
df_job_details["over_time"] = df_job_details["over_time"].astype(int)
df_career_history["attrition"] = df_career_history["attrition"].astype(int)

Crear tablas 

In [13]:
mycursor.execute(
    """CREATE TABLE IF NOT EXISTS job_details (
    job_id INT NOT NULL AUTO_INCREMENT,
    department VARCHAR(50) NOT NULL,
    job_role VARCHAR(50) NOT NULL,
    job_level INT NOT NULL,
    business_travel VARCHAR(50) NOT NULL,
    over_time BOOLEAN NOT NULL,
    PRIMARY KEY (job_id)
) ENGINE=InnoDB;""")

cnx.commit()
print("Tabla job_details creada")

Tabla job_details creada


In [14]:
mycursor.execute("""CREATE TABLE IF NOT EXISTS career_history (
    career_id INT NOT NULL AUTO_INCREMENT,
    total_working_years INT NOT NULL,
    years_at_company INT NOT NULL,
    years_in_current_role INT NOT NULL,
    years_since_last_promotion INT NOT NULL,
    years_with_curr_manager INT NOT NULL,
    num_companies_worked INT NOT NULL,
    training_times_last_year INT NOT NULL,
    attrition BOOLEAN NOT NULL,
    PRIMARY KEY (career_id)
) ENGINE=InnoDB;""")

cnx.commit()
print("Tabla career_history creada")

Tabla career_history creada


In [15]:
mycursor.execute("""CREATE TABLE IF NOT EXISTS compensation (
    compensation_id INT NOT NULL AUTO_INCREMENT,
    monthly_income INT NOT NULL,
    monthly_rate INT NOT NULL,
    percent_salary_hike INT NOT NULL,
    PRIMARY KEY (compensation_id)
) ENGINE=InnoDB;""")

cnx.commit()
print("Tabla compensation creada")

Tabla compensation creada


In [16]:
mycursor.execute("""CREATE TABLE IF NOT EXISTS performance_satisfaction (
    performance_satisfaction_id INT NOT NULL AUTO_INCREMENT,
    performance_rating INT NOT NULL,
    job_involvement INT NOT NULL,
    environment_satisfaction INT NOT NULL,
    relationship_satisfaction INT NOT NULL,
    job_satisfaction INT NOT NULL,
    work_life_balance INT NOT NULL,
    PRIMARY KEY (performance_satisfaction_id)
) ENGINE=InnoDB;""")

cnx.commit()
print("Tabla performance_satisfaction creada")

Tabla performance_satisfaction creada


In [17]:
mycursor.execute("""CREATE TABLE IF NOT EXISTS employees (
    employee_number VARCHAR(50) NOT NULL,
    age INT NOT NULL,
    gender VARCHAR(20) NOT NULL,
    marital_status VARCHAR(40) NOT NULL,
    education INT NOT NULL,
    education_field VARCHAR(70) NOT NULL,
    distance_from_home INT NOT NULL,
    job_id INT NOT NULL,
    career_id INT NOT NULL,
    compensation_id INT NOT NULL,
    performance_satisfaction_id INT NOT NULL,
    PRIMARY KEY (employee_number),
    FOREIGN KEY (job_id) REFERENCES job_details(job_id),
    FOREIGN KEY (career_id) REFERENCES career_history(career_id),
    FOREIGN KEY (compensation_id) REFERENCES compensation(compensation_id),
    FOREIGN KEY (performance_satisfaction_id) REFERENCES performance_satisfaction(performance_satisfaction_id)
) ENGINE=InnoDB;""")

cnx.commit()
print("Tabla employees creada")

Tabla employees creada


Insertar datos

In [18]:
def insertar_datos(cursor, conexion, query, dataframe, nombre_tabla):
    try:
        valores = []
        for fila in dataframe.itertuples(index=False, name=None):
            fila_limpia = tuple(x.item() if hasattr(x, "item") else x for x in fila)
            valores.append(fila_limpia)

        cursor.executemany(query, valores)
        conexion.commit()
        print(f"Datos insertados en {nombre_tabla}")

    except mysql.connector.Error as err:
        print(f"Error al insertar en {nombre_tabla}: {err}")

In [19]:
query_job = """INSERT INTO job_details (department,job_role,job_level,business_travel,over_time)
VALUES (%s, %s, %s, %s, %s)
"""

In [20]:
query_career = """INSERT INTO career_history (
    total_working_years,
    years_at_company,
    years_in_current_role,
    years_since_last_promotion,
    years_with_curr_manager,
    num_companies_worked,
    training_times_last_year,
    attrition)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"""

In [21]:
query_compensation = """INSERT INTO compensation (monthly_income, monthly_rate, percent_salary_hike)
    VALUES (%s, %s, %s)"""

In [22]:
query_performance = """INSERT INTO performance_satisfaction (
    performance_rating,
    job_involvement,
    environment_satisfaction,
    relationship_satisfaction,
    job_satisfaction,
    work_life_balance)
VALUES (%s, %s, %s, %s, %s, %s)"""

In [23]:
insertar_datos(mycursor, cnx, query_job, df_job_details, "job_details")
insertar_datos(mycursor, cnx, query_career, df_career_history, "career_history")
insertar_datos(mycursor, cnx, query_compensation, df_compensation, "compensation")
insertar_datos(mycursor,cnx,query_performance, df_performance_satisfaction,"performance_satisfaction")

Datos insertados en job_details


Datos insertados en career_history
Datos insertados en compensation
Datos insertados en performance_satisfaction


Recuperar tablas desde MySQL

In [24]:
def obtener_dataframe_desde_mysql(cursor, nombre_tabla, columnas):
    cursor.execute(f"SELECT * FROM {nombre_tabla}")
    resultados = cursor.fetchall()
    df_resultado = pd.DataFrame(resultados, columns=columnas)
    print(f"Datos recuperados de {nombre_tabla}")
    return df_resultado

In [25]:
df_job_details_ids = obtener_dataframe_desde_mysql(mycursor,"job_details",
    ["job_id", "department", "job_role", "job_level", "business_travel", "over_time"])

df_job_details_ids.head()

Datos recuperados de job_details


,job_id,department,job_role,job_level,business_travel,over_time
0,1,Sales,Sales Executive,2,Travel_Rarely,1
1,2,Research & Development,Research Scientist,2,Travel_Frequently,0
2,3,Research & Development,Laboratory Technician,1,Travel_Rarely,1
3,4,Research & Development,Research Scientist,1,Travel_Frequently,1
4,5,Research & Development,Laboratory Technician,1,Travel_Rarely,0


In [26]:
df_career_history_ids = obtener_dataframe_desde_mysql(mycursor,"career_history",
    ["career_id","total_working_years","years_at_company","years_in_current_role","years_since_last_promotion","years_with_curr_manager",
    "num_companies_worked","training_times_last_year","attrition"])

df_career_history_ids.head()


Datos recuperados de career_history


,career_id,total_working_years,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager,num_companies_worked,training_times_last_year,attrition
0,1,8,6,4,0,5,8,0,1
1,2,10,10,7,1,7,1,3,0
2,3,7,0,0,0,0,6,3,1
3,4,8,8,7,3,0,1,3,0
4,5,6,2,2,2,2,9,3,0


In [27]:
df_compensation_ids = obtener_dataframe_desde_mysql(mycursor,"compensation",
    ["compensation_id", "monthly_income", "monthly_rate", "percent_salary_hike"])

df_compensation_ids.head()

Datos recuperados de compensation


,compensation_id,monthly_income,monthly_rate,percent_salary_hike
0,1,5993,19479,11
1,2,5130,24907,23
2,3,2090,2396,15
3,4,2909,23159,11
4,5,3468,16632,12


In [28]:
df_performance_ids = obtener_dataframe_desde_mysql(mycursor,"performance_satisfaction",
    [ "performance_satisfaction_id","performance_rating","job_involvement", "environment_satisfaction","relationship_satisfaction","job_satisfaction",
        "work_life_balance"],
)

df_performance_ids.head()

Datos recuperados de performance_satisfaction


,performance_satisfaction_id,performance_rating,job_involvement,environment_satisfaction,relationship_satisfaction,job_satisfaction,work_life_balance
0,1,3,3,2,1,4,1
1,2,4,2,3,4,2,3
2,3,3,2,4,2,3,3
3,4,3,3,4,3,3,3
4,5,3,3,1,4,2,3


In [29]:
# comprobar como han vuelto los datos booleanos
print(df_job_details_ids.dtypes)
print(df_job_details_ids["over_time"].unique())

print(df_career_history_ids.dtypes)
print(df_career_history_ids["attrition"].unique())

job_id              int64
department         object
job_role           object
job_level           int64
business_travel    object
over_time           int64
dtype: object
[1 0]
career_id                     int64
total_working_years           int64
years_at_company              int64
years_in_current_role         int64
years_since_last_promotion    int64
years_with_curr_manager       int64
num_companies_worked          int64
training_times_last_year      int64
attrition                     int64
dtype: object
[1 0]


In [30]:
# pasamos a booleanos los datos
df_job_details_ids["over_time"] = df_job_details_ids["over_time"].astype(bool)
df_career_history_ids["attrition"] = df_career_history_ids["attrition"].astype(bool)

In [31]:
print(df_job_details_ids["over_time"].unique())
print(df_career_history_ids["attrition"].unique())

[ True False]
[ True False]


Hacemos merge para traer los IDs de las tablas auxiliares

In [32]:
def hacer_merge_ids(df_principal, df_ids, columnas_on, nombre_id):
    df_resultado = df_principal.merge(df_ids, on=columnas_on, how="left")
    print(f"Nulos en {nombre_id}: {df_resultado[nombre_id].isnull().sum()}")
    return df_resultado

In [33]:
def mergear_todos_los_ids(
    df_hr,
    df_job_details_ids,
    df_career_history_ids,
    df_compensation_ids,
    df_performance_ids,
):

    df_hr = hacer_merge_ids(
        df_hr,
        df_job_details_ids,
        ["department", "job_role", "job_level", "business_travel", "over_time"],
        "job_id",
    )

    df_hr = hacer_merge_ids(
        df_hr,
        df_career_history_ids,
        [
            "total_working_years",
            "years_at_company",
            "years_in_current_role",
            "years_since_last_promotion",
            "years_with_curr_manager",
            "num_companies_worked",
            "training_times_last_year",
            "attrition",
        ],
        "career_id",
    )

    df_hr = hacer_merge_ids(
        df_hr,
        df_compensation_ids,
        ["monthly_income", "monthly_rate", "percent_salary_hike"],
        "compensation_id",
    )

    df_hr = hacer_merge_ids(
        df_hr,
        df_performance_ids,
        [
            "performance_rating",
            "job_involvement",
            "environment_satisfaction",
            "relationship_satisfaction",
            "job_satisfaction",
            "work_life_balance",
        ],
        "performance_satisfaction_id",
    )

    return df_hr

In [34]:
df_hr = mergear_todos_los_ids(
    df_hr,
    df_job_details_ids,
    df_career_history_ids,
    df_compensation_ids,
    df_performance_ids,
)

Nulos en job_id: 0
Nulos en career_id: 0
Nulos en compensation_id: 0
Nulos en performance_satisfaction_id: 0


In [35]:
df_hr.head()

,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_number,environment_satisfaction,gender,...,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager,job_id,career_id,compensation_id,performance_satisfaction_id
0,41,True,Travel_Rarely,Sales,1,2,Life Sciences,1,2,Female,...,0,1,6,4,0,5,1,1,1,1
1,49,False,Travel_Frequently,Research & Development,8,1,Life Sciences,2,3,Male,...,3,3,10,7,1,7,2,2,2,2
2,37,True,Travel_Rarely,Research & Development,2,2,Other,4,4,Male,...,3,3,0,0,0,0,3,3,3,3
3,33,False,Travel_Frequently,Research & Development,3,4,Life Sciences,5,4,Female,...,3,3,8,7,3,0,4,4,4,4
4,27,False,Travel_Rarely,Research & Development,2,1,Medical,7,1,Male,...,3,3,2,2,2,2,5,5,5,5


Crear DATAFRAME empleados e insertamos los datos en la tabla de empleados

In [36]:
df_employees = df_hr[
    [ "employee_number",
        "age",
        "gender",
        "marital_status",
        "education",
        "education_field",
        "distance_from_home",
        "job_id",
        "career_id",
        "compensation_id",
        "performance_satisfaction_id",]]

In [37]:
query_employees = """INSERT INTO employees (
    employee_number,
    age,
    gender,
    marital_status,
    education,
    education_field,
    distance_from_home,
    job_id,
    career_id,
    compensation_id,
    performance_satisfaction_id)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [38]:
insertar_datos(mycursor, cnx, query_employees, df_employees, "employees")

Datos insertados en employees


In [39]:
# Comprobar registros en employees
mycursor.execute("SELECT COUNT(*) FROM employees")
print("Número de registros en employees:", mycursor.fetchone()[0])

Número de registros en employees: 1470


In [40]:
# ver algunas filas de employees
mycursor.execute("SELECT * FROM employees LIMIT 5")
resultado = mycursor.fetchall()
for fila in resultado:
    print(fila) 

('1', 41, 'Female', 'Single', '2', 'Life Sciences', 1, 1, 1, 1, 1)
('10', 59, 'Female', 'Married', '3', 'Medical', 3, 5, 7, 7, 7)
('100', 35, 'Male', 'Single', '4', 'Marketing', 1, 19, 76, 77, 70)
('1001', 27, 'Female', 'Married', '4', 'Technical Degree', 16, 5, 690, 718, 145)
('1002', 45, 'Male', 'Married', '2', 'Life Sciences', 23, 120, 691, 719, 177)


In [41]:
# Cerramos la conexión
mycursor.close()
cnx.close()
print("Conexión cerrada.")

Conexión cerrada.
